# Evaluating Arrays

Arrays need an **alignment** step before scoring. The evaluator supports three strategies:

1. **Ordered** (default) -- Position matters. `"x-eval-align": {"ordered": true}`
2. **Key-field** -- match by a unique identifier field. Order doesn't matter.  `"x-eval-align": {"match_by": "key_field", "key": "name"}`
3. **Hungarian** -- optimal bipartite matching. Order doesn't matter, no key field needed, optimizes for best F1.     `"x-eval-align": {"match_by": "hungarian"}`

After alignment, each matched pair is scored recursively using the `items` schema.
Unmatched gold elements are **omissions**. Unmatched extracted elements are **hallucinations**.

## Data

One record with three process steps. Each extracted element is wrong in a different way,
so each alignment strategy recovers a different amount:

- `deposition` is in the same position as gold -- every strategy gets it.
- `etch` is correct but moved -- only position-based matching misses it.
- `heating` is moved *and* named differently from gold's `anneal` -- only a strategy that
  looks past the name can pair it.


In [1]:
GOLD = [{"steps": [
    {"name": "deposition", "temp": 300},
    {"name": "anneal",     "temp": 500},
    {"name": "etch",       "temp": 25},
]}]

EXTRACTED = [{"steps": [
    {"name": "deposition", "temp": 300},   # same position as gold
    {"name": "etch",       "temp": 25},    # correct, but swapped with anneal
    {"name": "heating",    "temp": 500},   # swapped, and named differently
]}]


In [2]:
from struct_extract_eval import evaluate
from example_utils import show_run


## Strategy 1: Ordered (default)

Pairs by position: No `x-eval-align` needed -- this is the default when the key is absent.

e.g. time series, ordered instructions.

In [3]:
ORDERED_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            # No x-eval-align -> ordered (positional) matching
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

run_ordered = evaluate(GOLD, EXTRACTED, schema=ORDERED_SCHEMA)
show_run(run_ordered, "Ordered")

Ordered
  mean P=0.33  R=0.33  F1=0.33   (1 record(s))
  record  path           gold          extracted     score  status    reason
  0       steps[0].name  'deposition'  'deposition'  1.0    match
  0       steps[0].temp  300           300           1.0    match
  0       steps[1].name  'anneal'      'etch'        0.0    mismatch  mismatch
  0       steps[1].temp  500           25            0.0    mismatch  values differ
  0       steps[2].name  'etch'        'heating'     0.0    mismatch  mismatch
  0       steps[2].temp  25            500           0.0    mismatch  values differ


Only `steps[0]` lines up. Positions 1 and 2 are compared against the wrong elements, so
four of the six fields are mismatches even though `etch` is extracted perfectly.


## Strategy 2: Key-Field Matching

Match elements by the value of a unique identifier field. Add `x-eval-align` to the
array node:

```json
"x-eval-align": {"match_by": "key_field", "key": "name"}
```

Gold's `"deposit"` pairs with extracted's `"deposit"`, regardless of position.

Use when elements have a natural unique key (name, id, formula).

In [4]:
KEYFIELD_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "x-eval-align": {"match_by": "key_field", "key": "name"},
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

run_keyfield = evaluate(GOLD, EXTRACTED, schema=KEYFIELD_SCHEMA)
show_run(run_keyfield, "Key-field")

Key-field
  mean P=0.67  R=0.67  F1=0.67   (1 record(s))
  record  path            gold          extracted     score  status         reason
  0       steps[0].name   'deposition'  'deposition'  1.0    match
  0       steps[0].temp   300           300           1.0    match
  0       steps[1].name   'anneal'      None          0.0    omission
  0       steps[1].temp   500           None          0.0    omission
  0       steps[2].name   'etch'        'etch'        1.0    match
  0       steps[2].temp   25            25            1.0    match
  0       steps[-1].name  None          'heating'     0.0    hallucination
  0       steps[-1].temp  None          500           0.0    hallucination


Better. `etch` now pairs with `etch` regardless of position.

`anneal` still fails, and it fails twice: key-field matching pairs on the exact `name`,
so gold's `anneal` finds no partner and becomes two **omissions**, while extracted's
`heating` finds no partner and becomes two **hallucinations**. One unpaired element costs
both sides.

**Note**: a path like `steps[-1]` is an extracted element with no gold counterpart. Negative
indices count down (`-1`, `-2`, ...) so each hallucinated element keeps a distinct path.


## Strategy 3: Hungarian Matching

When elements have no unique key field, use Hungarian matching. It finds the optimal
pairing that maximizes total F1 across all pairs.

```json
"x-eval-align": {"match_by": "hungarian"}
```

In [5]:
HUNGARIAN_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "x-eval-align": {"match_by": "hungarian"},
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

run_hungarian = evaluate(GOLD, EXTRACTED, schema=HUNGARIAN_SCHEMA)
show_run(run_hungarian, "Hungarian")

Hungarian
  mean P=0.83  R=0.83  F1=0.83   (1 record(s))
  record  path           gold          extracted     score  status    reason
  0       steps[0].name  'deposition'  'deposition'  1.0    match
  0       steps[0].temp  300           300           1.0    match
  0       steps[1].name  'anneal'      'heating'     0.0    mismatch  mismatch
  0       steps[1].temp  500           500           1.0    match
  0       steps[2].name  'etch'        'etch'        1.0    match
  0       steps[2].temp  25            25            1.0    match


Best of the three. Hungarian pairs gold's `anneal` step with extracted's `heating` step
because their `temp` values match, so only the `name` field is lost instead of the whole element.

## Other Example: Hungarian with Primitive Arrays

Hungarian shines with arrays of primitives where key-field matching isn't possible.

In [6]:
TAGS_GOLD = [{"tags": ["silicon", "thin-film", "CVD"]}]
TAGS_EXTRACTED = [{"tags": ["CVD", "silicon", "PVD"]}]

TAGS_ORDERED = {
    "type": "object",
    "properties": {
        "tags": {
            "type": "array",
            "items": {"type": "string", "x-eval-compare": "exact"},
        },
    },
}

TAGS_HUNGARIAN = {
    "type": "object",
    "properties": {
        "tags": {
            "type": "array",
            "x-eval-align": {"match_by": "hungarian"},
            "items": {"type": "string", "x-eval-compare": "exact"},
        },
    },
}

run_tags_ord = evaluate(TAGS_GOLD, TAGS_EXTRACTED, schema=TAGS_ORDERED)
run_tags_hun = evaluate(TAGS_GOLD, TAGS_EXTRACTED, schema=TAGS_HUNGARIAN)

print(f"Ordered:   F1={run_tags_ord.mean_f1:.3f}  (positional: 'silicon' vs 'CVD' = mismatch)")
print(f"Hungarian: F1={run_tags_hun.mean_f1:.3f}  ('silicon' matches 'silicon', 'CVD' matches 'CVD')")
print()
show_run(run_tags_hun)

Ordered:   F1=0.000  (positional: 'silicon' vs 'CVD' = mismatch)
Hungarian: F1=0.667  ('silicon' matches 'silicon', 'CVD' matches 'CVD')

  mean P=0.67  R=0.67  F1=0.67   (1 record(s))
  record  path      gold         extracted  score  status         reason
  0       tags[0]   'silicon'    'silicon'  1.0    match
  0       tags[2]   'CVD'        'CVD'      1.0    match
  0       tags[1]   'thin-film'  None       0.0    omission
  0       tags[-1]  None         'PVD'      0.0    hallucination
